# Module 6 — RDDs and mapPartitions

**What you will learn:**
- What RDDs are and how they differ from DataFrames
- How to use `filter`, `map`, and `reduceByKey` on RDDs
- How to convert RDD results back into a DataFrame using a schema
- How `mapPartitions` works for processing entire partitions at once
- How to use `mapPartitions` to apply a machine learning model in batch

**Before you run this notebook:**
- Make sure `03 taxi_schema.ipynb` has been run so `data/pq/green/` exists
- Make sure `spark_basics.ipynb` has been run so `zones/` exists

---
## Part 1 — Operations on Spark RDDs

An **RDD** (Resilient Distributed Dataset) is the low-level building block of Spark.
DataFrames are built on top of RDDs. You can convert between them.

When would you use RDDs?
- When you need custom logic that SQL and DataFrame API cannot express
- When you want to understand what Spark is doing internally

## Step 1 — Start Spark Session

In [ ]:
# What happens: Spark engine starts on your local machine using all CPU cores.
# setLogLevel("ERROR") hides INFO and WARN messages so the output stays clean.
# Check Dashboard: Open http://localhost:4040 — you should see Spark is running.
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

## Step 2 — Read Green Taxi Parquet Files

In [ ]:
# What happens: Loads all green taxi parquet files across all months into a DataFrame.
# No Spark job runs yet — this is lazy. Spark only reads the file metadata (schema).
# The ** means "match any folder" — it loads all years and months at once.
df_green = spark.read.parquet('data/pq/green/*/*')

print("Columns:", df_green.columns)
print("Row count (triggers job):", df_green.count())

## Step 3 — Select Columns and Convert DataFrame to RDD

A **DataFrame** has a schema (column names and types). An **RDD** is just a distributed
collection of objects — like a Python list but split across many machines.

We only need 3 columns for this calculation, so we select them first.

In [ ]:
# What happens: Selects 3 columns from the DataFrame, then converts to RDD.
# Each row in the RDD is a Row object — like a Python namedtuple.
# You can access fields by name: row.lpep_pickup_datetime, row.PULocationID
rdd = df_green \
    .select('lpep_pickup_datetime', 'PULocationID', 'total_amount') \
    .rdd

# Peek at the first 5 rows — this triggers a small Spark job
# Check Dashboard: http://localhost:4040 -> Jobs -> you will see a new job appear
rdd.take(5)

## Step 4 — Define the Filter Function

We only want trips from **2020 onwards**. We define a Python function that returns
`True` (keep) or `False` (discard) for each row.

This is like a SQL `WHERE` clause but written as Python code.

In [ ]:
# What happens: Defines a cutoff date and a filter function.
# filter_outliers(row) returns True if the trip is on or after Jan 1 2020.
# False means "throw away this row".
# No Spark job runs yet — this is just Python function definition.
from datetime import datetime

start = datetime(year=2020, month=1, day=1)

def filter_outliers(row):
    return row.lpep_pickup_datetime >= start

# Test it on the first 5 rows to verify it works
for row in rdd.take(5):
    print(row.lpep_pickup_datetime, "| Keep?", filter_outliers(row))

## Step 5 — Define the Map Function (Prepare Key-Value Pairs)

For `reduceByKey` to work, each row must become a **(key, value)** pair.

- **Key** = `(hour, zone)` — what we are grouping by
- **Value** = `(total_amount, count=1)` — what we are aggregating

Spark will then group all rows with the same key and reduce the values.

In [ ]:
# What happens: Transforms each row into a (key, value) tuple.
# hour = the pickup time rounded down to the nearest hour (minutes/seconds set to 0)
# zone = the pickup location ID
# key = (hour, zone) — the thing we are grouping by
# value = (total_amount, 1) — amount for this trip + count of 1
# No Spark job runs yet — just defining the transformation.
def prepare_for_grouping(row):
    hour = row.lpep_pickup_datetime.replace(minute=0, second=0, microsecond=0)
    zone = row.PULocationID
    key = (hour, zone)
    
    amount = row.total_amount
    count = 1
    value = (amount, count)
    
    return (key, value)

# Test it on one row to verify the output format
sample_row = rdd.take(1)[0]
print("Input row:", sample_row)
print("Output (key, value):", prepare_for_grouping(sample_row))

## Step 6 — Define the ReduceByKey Function (Aggregate Values)

`reduceByKey` takes two values with the same key and combines them.
It runs repeatedly until all values for the same key are merged into one.

Think of it like: for each `(hour, zone)` group, add up all amounts and counts.

In [ ]:
# What happens: Defines how to combine two values for the same key.
# left_value = (amount_so_far, count_so_far)
# right_value = (this_trip_amount, 1)
# output = (total_amount, total_count)
# Spark calls this function many times to merge all trips in the same group.
def calculate_revenue(left_value, right_value):
    left_amount, left_count = left_value
    right_amount, right_count = right_value
    
    output_amount = left_amount + right_amount
    output_count = left_count + right_count
    
    return (output_amount, output_count)

# Test it manually with two fake values
v1 = (100.0, 5)
v2 = (50.0, 3)
print("Merged:", calculate_revenue(v1, v2))  # should be (150.0, 8)

## Step 7 — Define namedtuple for Column Names

After `reduceByKey`, each row looks like: `((hour, zone), (revenue, count))`

This is hard to read and cannot be directly converted to a DataFrame with column names.
We use Python's `namedtuple` to give the fields names — like a struct or a lightweight class.

In [ ]:
# What happens: Creates a namedtuple class called RevenueRow.
# It has 4 named fields: hour, zone, revenue, count.
# This makes it easier to convert back to a DataFrame with proper column names.
from collections import namedtuple

RevenueRow = namedtuple('RevenueRow', ['hour', 'zone', 'revenue', 'count'])

# Test: create one sample row
sample = RevenueRow(hour=start, zone=43, revenue=150.0, count=8)
print("Sample row:", sample)
print("Access by name:", sample.revenue, sample.count)

## Step 8 — Define Unwrap Function

After `reduceByKey`, each row has this shape: `((hour, zone), (revenue, count))`

The `unwrap` function flattens this into a `RevenueRow` with 4 named fields.

In [ ]:
# What happens: Converts the nested tuple structure into a flat RevenueRow.
# Input:  ((hour, zone), (revenue, count))
# Output: RevenueRow(hour=..., zone=..., revenue=..., count=...)
# row[0] = the key = (hour, zone)
# row[1] = the value = (revenue, count)
def unwrap(row):
    return RevenueRow(
        hour=row[0][0],
        zone=row[0][1],
        revenue=row[1][0],
        count=row[1][1]
    )

# Test it on a fake reduced row
fake_reduced_row = ((start, 43), (150.0, 8))
print("Unwrapped:", unwrap(fake_reduced_row))

## Step 9 — Define the Output Schema

When converting an RDD back to a DataFrame, Spark needs to know the **data types** of each column.
We define a `StructType` schema to tell Spark exactly what the columns are.

In [ ]:
# What happens: Defines the schema for the result DataFrame.
# This tells Spark: hour=timestamp, zone=integer, revenue=double, count=integer.
# Without this schema, Spark would guess the types and might get them wrong.
from pyspark.sql import types

result_schema = types.StructType([
    types.StructField('hour', types.TimestampType(), True),
    types.StructField('zone', types.IntegerType(), True),
    types.StructField('revenue', types.DoubleType(), True),
    types.StructField('count', types.IntegerType(), True)
])

print("Schema defined:", result_schema)

## Step 10 — Run the Full RDD Pipeline

Now we chain all steps together:
1. `filter` — remove trips before 2020
2. `map` — convert each row to (key, value) pairs
3. `reduceByKey` — group by key and aggregate
4. `map` — flatten nested tuple into named row
5. `toDF` — convert RDD back to DataFrame with schema

**This is still lazy** — nothing runs until we trigger an action.

In [ ]:
# What happens: Chains all 4 RDD operations into a pipeline.
# Still lazy at this point — Spark builds a plan but does not execute yet.
# toDF(result_schema) wraps the RDD back into a DataFrame.
df_result = rdd \
    .filter(filter_outliers) \
    .map(prepare_for_grouping) \
    .reduceByKey(calculate_revenue) \
    .map(unwrap) \
    .toDF(result_schema)

# Preview the result — this triggers the Spark job
# Check Dashboard: http://localhost:4040 -> Jobs -> click the running job
# -> Stages: you should see 2 stages (Stage 1: per-partition grouping, Stage 2: final merge)
df_result.show(5)

## Step 11 — Write RDD Result to Parquet

Now we save the result to disk as a Parquet file.
This is the action that forces the full pipeline to run if it hasn't already.

In [ ]:
# What happens: Runs the full RDD pipeline and saves result to Parquet.
# mode('overwrite') means it will replace any existing files at this path.
# Check Dashboard: http://localhost:4040 -> Jobs -> click the write job
# -> Stages: Stage 1 filters + maps, Stage 2 does reduceByKey (requires shuffle)
df_result.write.mode('overwrite').parquet('tmp/green-revenue')

print("Done! Result saved to tmp/green-revenue")

## Step 12 — Verify the Saved Result

In [ ]:
# What happens: Reads back the saved parquet and checks the schema and row count.
# This confirms the file was written correctly.
df_check = spark.read.parquet('tmp/green-revenue')
df_check.printSchema()
print("Total rows:", df_check.count())
df_check.show(10)

---
## Part 2 — mapPartitions for ML Batch Prediction

`mapPartitions` is like `map` but instead of processing **one row at a time**,
it processes an **entire partition** at a time.

**Why use mapPartitions?**
- Loading a machine learning model is expensive. You only want to load it **once per partition**, not once per row.
- With `map`, the model would be re-loaded for every single row.
- With `mapPartitions`, the model is loaded once and then used for all rows in that partition.

In this example, we simulate an ML model that predicts trip duration from trip distance.

## Step 13 — Select Columns and Build the Prediction RDD

In [ ]:
# What happens: Selects 5 columns needed for prediction and converts to RDD.
# Each row in the RDD will be passed to our prediction function.
# We save the column names in a list so we can use them inside mapPartitions.
columns = ['VendorID', 'lpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance']

duration_rdd = df_green.select(columns).rdd

# Peek at the first 3 rows to see the data format
duration_rdd.take(3)

## Step 14 — Import pandas

Inside `mapPartitions`, we will convert the partition rows into a **pandas DataFrame**.
This lets us apply the ML model to a whole batch of rows at once — much faster than row-by-row.

In [ ]:
# What happens: Imports pandas for batch processing inside mapPartitions.
# pandas DataFrames are in-memory (not distributed) and live on one executor.
# Each Spark partition becomes one pandas DataFrame when we call pd.DataFrame(rows).
import pandas as pd

print("pandas version:", pd.__version__)

## Step 15 — Define a Mock ML Model

In a real project, this function would load a trained model (sklearn, XGBoost, etc.)
and call `.predict()`. Here we simulate it with a simple formula:
`predicted_duration = trip_distance * 5`

In [ ]:
# What happens: Defines a fake ML model function.
# In a real scenario this would be: model = joblib.load('model.pkl'); model.predict(df)
# Here we use a simple formula: predicted trip duration = distance * 5 minutes per mile.
# The function takes a pandas DataFrame and returns a pandas Series of predictions.
def model_predict(df):
    y_pred = df.trip_distance * 5
    return y_pred

# Test it on a small pandas DataFrame
test_df = pd.DataFrame({'trip_distance': [1.0, 2.5, 0.5]})
print("Test predictions:", model_predict(test_df).tolist())

## Step 16 — Define the mapPartitions Function

This is the key function. It receives an **iterator** of rows (the whole partition),
converts them to a pandas DataFrame, runs the model, and **yields** back the results.

`yield` means the function returns results one row at a time without loading everything into memory.
This is called a **generator** — efficient for large datasets.

In [ ]:
# What happens: Defines the mapPartitions function.
# 'rows' = an iterator of Row objects for one entire partition
# Step A: Convert partition rows to a pandas DataFrame (all rows in memory at once)
# Step B: Run the ML model on the whole batch — fast vectorized operation
# Step C: Add the predictions as a new column
# Step D: Use 'yield' to return each row one at a time (memory efficient)
# 'yield' vs 'return': return sends everything at once, yield sends one row at a time
def apply_model_in_batch(rows):
    # Step A: All rows in this partition -> pandas DataFrame
    df = pd.DataFrame(rows, columns=columns)
    
    # Step B: Run ML model on the batch
    predictions = model_predict(df)
    
    # Step C: Add prediction column
    df['predicted_duration'] = predictions
    
    # Step D: Yield each row back to Spark one at a time
    for row in df.itertuples():
        yield row

print("Function defined. It uses 'yield' so it is a Python generator.")

## Step 17 — Run mapPartitions and Show Predictions

Now we apply `apply_model_in_batch` to every partition of the RDD.
Spark will call this function once per partition — loading the model once per partition.

In [ ]:
# What happens: Applies the mapPartitions function across all partitions.
# Each executor gets one partition at a time and calls apply_model_in_batch.
# The result is converted back to a Spark DataFrame with .toDF()
# .drop('Index') removes the pandas row index column that itertuples() adds.
# Check Dashboard: http://localhost:4040 -> Jobs -> click the running job
# -> Stages: should show 1 stage (no shuffle needed — each partition is processed independently)
df_predicts = duration_rdd \
    .mapPartitions(apply_model_in_batch) \
    .toDF() \
    .drop('Index')

# Show the predicted duration column
df_predicts.select('trip_distance', 'predicted_duration').show(10)

## Step 18 — Show the Full Schema of the Prediction Result

In [ ]:
# What happens: Prints the schema of the prediction result DataFrame.
# You should see all original columns plus 'predicted_duration' at the end.
# Notice the types — Spark inferred them from the pandas DataFrame.
df_predicts.printSchema()

---
## Summary — What You Learned

| Concept | What it does |
|---|---|
| `.rdd` | Converts a DataFrame to a low-level RDD |
| `.filter(fn)` | Keeps rows where fn returns True |
| `.map(fn)` | Transforms each row using fn — 1 row in, 1 row out |
| `.reduceByKey(fn)` | Groups rows by key and merges values using fn |
| `namedtuple` | Gives field names to plain tuples so DataFrame conversion works |
| `.toDF(schema)` | Converts RDD back to DataFrame with correct column types |
| `.mapPartitions(fn)` | Like map but fn receives the whole partition — load model once |
| `yield` | Returns rows one at a time from a generator — memory efficient |

**When to use RDDs vs DataFrames:**
- Use **DataFrames** for most SQL-style queries — they are faster and easier
- Use **RDDs** when you need custom Python logic that DataFrames cannot express
- Use **mapPartitions** specifically when loading an ML model or database connection per partition